> ☁️ **This notebook runs against the free shared sandbox** — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")`. It uses features that need a server (fleet delegation, category allocations, shadow mode, or cross-process budgets). The sandbox is for evaluation only — **not for production** (may be wiped at any time; no SLA).
>
> **Three ways to run FiGuard — same API, same rules. Swap only the client constructor:**
>
> - ☁️ **Free sandbox** _(this notebook)_ — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")` — quick hosted evaluation.
> - 🏢 **Self-hosted server** — `FiGuardClient(base_url="https://your-figuard", api_key="...")` — your data, your infra; the production path for the server features this notebook uses. [Self-host →](https://github.com/figuard/figuard-core#self-hosting)
> - 💻 **Embedded** — `FiGuardClient()` — in-process, zero infra (single process; the server-only features used here aren't available embedded).

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys, importlib
!pip install "figuard>=0.3.0" --upgrade -q
# Flush any stale cached module from this runtime session
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")
import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (identifies your budgets in the dashboard)")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")

budget = client.create_budget(
    user_id=USER_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit:   ${budget.total_limit}")
print(f"  Session token: {budget.primary_token.session_token[:20]}...")

session_token = budget.session_token

## Velocity controls — stopping runaway loops

`velocity_max_per_minute` caps the number of `authorize` calls in any rolling 60-second window. This catches retry loops before they accumulate meaningful spend — even if the budget is already exhausted and every call returns `BUDGET_EXHAUSTED`, the velocity counter still increments on each attempt.

Set it on `create_budget` alongside your other safety controls. It can also be updated later via `PATCH /budgets/{id}` without recreating the budget.

In [ ]:
budget_with_velocity = client.create_budget(
    user_id=USER_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session — velocity controls demo",
    velocity_max_per_minute=5,
)

print(f"✓ Budget created: {budget_with_velocity.id}")
print(f"  velocity_max_per_minute=5  →  6th call in any 60-second window is denied\n")

velocity_token = budget_with_velocity.primary_token.session_token

for i in range(1, 7):
    r = client.authorize(
        session_token=velocity_token,
        agent_id="retry_loop_agent",
        action_type="PURCHASE",
        description=f"Retry attempt {i}",
        requested_quantity=10.00,
        idempotency_key=f"velocity-demo-{budget_with_velocity.id}-{i}",
    )
    label = r.decision if r.is_authorized else r.denial_reason
    print(f"  Call {i}: {label}")

print("\nCall 6 hit VELOCITY_LIMIT_EXCEEDED — runaway loop stopped before accumulating more spend.")
print(f"\n→ See the ledger: https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{budget_with_velocity.id}")


In [ ]:
from figuard import clear_current_event_id
clear_current_event_id()  # isolate from velocity demo
# Authorized transaction
auth = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="JetBlue SFO→JFK roundtrip",
    requested_quantity=270.00,
    idempotency_key="booking-001",
)
print(f"authorize() →  AUTHORIZED    ${auth.approved_quantity:.2f} reserved  (nothing moved yet)")

client.confirm_event(auth.event_id, confirmed_quantity=267.00)
print(f"confirm()   →  CONFIRMED     $267.00 settled  (reservation released)")

# Denied — over remaining budget
auth2 = client.authorize(
    session_token=session_token,
    agent_id="travel_agent",
    action_type="PURCHASE",
    description="Business class upgrade",
    requested_quantity=890.00,
    idempotency_key="booking-002",
)
print(f"authorize() →  DENIED        {auth2.denial_reason}  (nothing moved)")
print("Nothing moved. Ledger recorded the denial.")

print(f"\n→ Main budget:     https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{budget.id}")
print(f"→ Velocity budget: https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{budget_with_velocity.id}")

## Shadow mode — observe before enforcing

Not sure what limits to set? Run your agent in shadow mode first.
All enforcement checks run, nothing gets blocked, and every `authorize` response
tells you what *would have* happened under full enforcement.
When the patterns look right, flip to `FULL_ENFORCEMENT` with a single PATCH.

In [ ]:
clear_current_event_id()  # isolate from previous section
from figuard import FiGuardClient

client = FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")

# Budget is in shadow mode — all checks run, nothing is blocked
shadow_budget = client.create_budget(
    user_id=USER_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    trust_mode="SHADOW",
)

print(f"✓ Budget created in shadow mode: {shadow_budget.id}")
print(f"  trust_mode: {shadow_budget.trust_mode}")

# This would be DENIED (budget exhausted) in full enforcement,
# but is AUTHORIZED here because the budget is in shadow mode.
auth = client.authorize(
    session_token=shadow_budget.primary_token.session_token,
    agent_id="refund_agent",
    action_type="REFUND",
    description="Shadow refund — $1,200 would be denied in full enforcement",
    requested_quantity=1_200.00,
    idempotency_key="refund-shadow-001",
)

print(f"\nAuthorize response:")
print(f"  decision:              {auth.decision}")
print(f"  shadow:                {auth.shadow}")
print(f"  would_have_been:       {auth.would_have_been}")
print(f"  would_have_been_reason:{auth.would_have_been_reason}")

# Once you're confident in the limits, flip to full enforcement:
# client.update_budget(shadow_budget.id, trust_mode="FULL_ENFORCEMENT")
# print("Flipped to FULL_ENFORCEMENT — agent is now fully guarded.")

print(f"\n→ See the shadow budget in the dashboard: https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{shadow_budget.id}")